# Train Query-to-Graph Projection

This notebook trains a query projection into the capsule space produced by the R-GCN notebook. MetaQA uses one unified projector trained across mixed 1/2/3-hop questions; hop labels are never required by runtime retrieval.

In [ ]:
from __future__ import annotations

import hashlib
import json
import random
import re
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.adapters.datasets import get_dataset_adapter
from src.adapters.embedding import CachingEmbeddingAdapter, OllamaEmbeddingAdapter, OllamaEmbeddingConfig, local_embedding_cache_namespace
from src.optimized_grag.models import mine_metaqa_path_projection_examples, mine_projection_examples

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOKEN_RE = re.compile(r"[A-Za-z0-9_]+")

OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen3:4b-instruct"
USE_OLLAMA_QUERY_EMBEDDINGS = True
EMBED_BATCH_SIZE = 16  # lower to 8 if Ollama embedding itself approaches VRAM capacity
CAPSULE_SCORE_BATCH_SIZE = 2_048
base_embedding_adapter = OllamaEmbeddingAdapter(
    OllamaEmbeddingConfig(
        model=EMBEDDING_MODEL,
        base_url=OLLAMA_BASE_URL,
        keep_alive="30m",
        options={"num_gpu": 999},
        timeout_seconds=180.0,
        batch_size=EMBED_BATCH_SIZE,
    )
)
embedding_adapter = CachingEmbeddingAdapter(
    base_embedding_adapter,
    cache_path=ROOT / "artifacts" / "cache" / "embeddings.sqlite3",
    namespace=local_embedding_cache_namespace(EMBEDDING_MODEL),
    model_name=EMBEDDING_MODEL,
)
device

## Choose One Training Dataset

The next three cells define configurations without activating them. Set `PROJECTION_DATASET` in the final configuration cell; `Run All` is safe.

To train all three separate models, run the notebook once for MetaQA, once for 2WikiMultiHopQA, and once for MultiHop-RAG. Each dataset writes into its own artifact folder.

In [ ]:
METAQA_DATASET = {
    "label": "MetaQA unified 1/2/3-hop vanilla train",
    "artifact_name": "metaqa_unified_vanilla_train",
    "structural_artifact_name": "metaqa_shared_graph_train",
    "adapter_name": "metaqa",
    "training_slices": [
        {"hop": "1-hop", "variant": "vanilla", "split": "train"},
        {"hop": "2-hop", "variant": "vanilla", "split": "train"},
        {"hop": "3-hop", "variant": "vanilla", "split": "train"},
    ],
    "limit_documents": 1,  # projection needs questions, not three copies of the shared KB
    "limit_eval_samples_per_slice": 5_000,
    "graph_from_documents": False,
    "chunk_size": 900,
    "chunk_overlap": 120,
    "max_projection_examples": 30_000,
    "max_positive_capsules": 16,
    "max_hard_negatives": 20,
    "capsule_config": {
        "max_region_nodes": 1000,
        "max_capsule_nodes": 250,
        "max_capsule_edges": 900,
        "max_hops": 2,
        "anchors_per_region": 5,
        "max_capsules_per_region": 5,
    },
}
METAQA_DATASET

In [ ]:
WIKI_DATASET = {
    "label": "2WikiMultiHopQA train sample",
    "artifact_name": "2wiki_train_sample",
    "adapter_name": "2wiki",
    "adapter_kwargs": {"split": "train"},
    "limit_documents": 1_200,
    "limit_eval_samples": 1_200,
    "graph_from_documents": True,
    "chunk_size": 850,
    "chunk_overlap": 100,
    "max_projection_examples": 8_000,
    "max_positive_capsules": 16,
    "max_hard_negatives": 20,
    "capsule_config": {
        "max_region_nodes": 700,
        "max_capsule_nodes": 180,
        "max_capsule_edges": 650,
        "max_hops": 2,
        "anchors_per_region": 4,
        "max_capsules_per_region": 4,
    },
}
WIKI_DATASET

In [ ]:
MULTIHOP_RAG_DATASET = {
    "label": "MultiHop-RAG train sample",
    "artifact_name": "multihop_rag_train_sample",
    "adapter_name": "multihop_rag",
    "adapter_kwargs": {"split": "train"},
    "limit_documents": 1_500,
    "limit_eval_samples": 1_500,
    "graph_from_documents": True,
    "chunk_size": 850,
    "chunk_overlap": 100,
    "max_projection_examples": 8_000,
    "max_positive_capsules": 16,
    "max_hard_negatives": 20,
    "capsule_config": {
        "max_region_nodes": 700,
        "max_capsule_nodes": 180,
        "max_capsule_edges": 650,
        "max_hops": 2,
        "anchors_per_region": 4,
        "max_capsules_per_region": 4,
    },
}
PROJECTION_DATASETS = {
    "metaqa": METAQA_DATASET,
    "2wiki": WIKI_DATASET,
    "multihop_rag": MULTIHOP_RAG_DATASET,
}
PROJECTION_DATASET = "metaqa"  # choose: metaqa, 2wiki, multihop_rag
ACTIVE_DATASET = PROJECTION_DATASETS[PROJECTION_DATASET]
ACTIVE_DATASET

In [ ]:
if "ACTIVE_DATASET" not in globals():
    raise RuntimeError("Run one projection training selection cell first.")

if PROJECTION_DATASET not in PROJECTION_DATASETS:
    raise ValueError(f"Unknown PROJECTION_DATASET={PROJECTION_DATASET!r}; choose one of {sorted(PROJECTION_DATASETS)}")

EXPORT_DIR = ROOT / "artifacts" / "rgcn" / ACTIVE_DATASET["artifact_name"]
STRUCTURAL_DIR = ROOT / "artifacts" / "rgcn" / ACTIVE_DATASET.get("structural_artifact_name", ACTIVE_DATASET["artifact_name"])
rgcn_metadata_path = STRUCTURAL_DIR / "capsule_rgcn_metadata.json"
if not rgcn_metadata_path.exists():
    available = sorted(path.parent.name for path in (ROOT / "artifacts" / "rgcn").glob("*/capsule_rgcn_metadata.json"))
    selectable = {
        name: config.get("structural_artifact_name", config["artifact_name"])
        for name, config in PROJECTION_DATASETS.items()
        if (ROOT / "artifacts" / "rgcn" / config.get("structural_artifact_name", config["artifact_name"]) / "capsule_rgcn_metadata.json").exists()
    }
    raise FileNotFoundError(
        f"Missing R-GCN metadata for selected projection dataset={PROJECTION_DATASET!r} "
        f"at {rgcn_metadata_path}. Train that dataset's R-GCN first or set "
        f"PROJECTION_DATASET to one of {sorted(selectable)}. Available structural artifacts={available}"
    )
rgcn_metadata = json.loads(rgcn_metadata_path.read_text(encoding="utf-8"))
projection_manifest = str(rgcn_metadata.get("projection_manifest") or "")
if not projection_manifest:
    raise RuntimeError("R-GCN artifact has no projection manifest; run the repool utility or updated R-GCN notebook.")
embedding_path = STRUCTURAL_DIR / projection_manifest.split(":", 1)[0]
if not embedding_path.exists():
    raise FileNotFoundError(f"Projection manifest points to missing file: {embedding_path}")
expected_capsule_fingerprint = rgcn_metadata.get("capsule_fingerprint")
if not expected_capsule_fingerprint:
    raise RuntimeError("R-GCN artifact has no capsule fingerprint; retrain it with the updated notebook.")
rgcn_export = np.load(embedding_path, allow_pickle=True)
capsule_ids = [str(x) for x in rgcn_export["capsule_ids"]]
capsule_embeddings = rgcn_export["embeddings"].astype("float32")
if "search_texts" not in rgcn_export.files:
    raise RuntimeError("R-GCN artifact has no projection manifest; rerun the updated R-GCN notebook.")
capsule_search_texts = [str(value) for value in rgcn_export["search_texts"]]
if len(capsule_search_texts) != len(capsule_ids):
    raise RuntimeError("R-GCN projection manifest is not aligned with capsule embeddings.")
capsule_to_idx = {capsule_id: idx for idx, capsule_id in enumerate(capsule_ids)}
path_supervision_fields = {"capsule_node_ids", "capsule_edge_ids", "graph_edge_ids", "graph_edge_sources", "graph_edge_targets"}
has_path_supervision = path_supervision_fields.issubset(rgcn_export.files)
capsule_edge_ids_by_id = (
    {capsule_id: [str(value) for value in edge_ids] for capsule_id, edge_ids in zip(capsule_ids, rgcn_export["capsule_edge_ids"])}
    if has_path_supervision else {}
)
capsule_node_ids_by_id = (
    {capsule_id: [str(value) for value in node_ids] for capsule_id, node_ids in zip(capsule_ids, rgcn_export["capsule_node_ids"])}
    if has_path_supervision else {}
)
projection_graph_edges = (
    [(str(edge_id), str(source), str(target)) for edge_id, source, target in zip(rgcn_export["graph_edge_ids"], rgcn_export["graph_edge_sources"], rgcn_export["graph_edge_targets"])]
    if has_path_supervision else []
)
graph_dim = capsule_embeddings.shape[1]
safe_embedding_model = "".join(character if character.isalnum() else "_" for character in EMBEDDING_MODEL)
semantic_target_path = EXPORT_DIR / f"capsule_semantic_embeddings_{safe_embedding_model}.npz"
if semantic_target_path.exists():
    semantic_export = np.load(semantic_target_path, allow_pickle=True)
    semantic_capsule_ids = [str(value) for value in semantic_export["capsule_ids"]]
    if semantic_capsule_ids != capsule_ids:
        raise RuntimeError(
            "Semantic capsule target IDs do not align with the structural manifest. "
            f"Rebuild it with: python scripts/build_capsule_semantic_artifact.py --structural-artifact {STRUCTURAL_DIR.name} --projector-artifact {EXPORT_DIR.name}"
        )
    semantic_fingerprint = str(semantic_export["capsule_fingerprint"].item()) if "capsule_fingerprint" in semantic_export.files else ""
    if semantic_fingerprint and semantic_fingerprint != expected_capsule_fingerprint:
        raise RuntimeError(
            "Semantic capsule target fingerprint does not match the selected structural R-GCN artifact. "
            f"Rebuild it with: python scripts/build_capsule_semantic_artifact.py --structural-artifact {STRUCTURAL_DIR.name} --projector-artifact {EXPORT_DIR.name}"
        )
    capsule_target_embeddings = semantic_export["embeddings"].astype("float32")
    target_space = "semantic_capsule_text"
    target_manifest = semantic_target_path.name
else:
    capsule_target_embeddings = capsule_embeddings
    target_space = "structural_rgcn"
    target_manifest = projection_manifest.split(":", 1)[0]
target_dim = capsule_target_embeddings.shape[1]
{
    "structural_dir": STRUCTURAL_DIR,
    "export_dir": EXPORT_DIR,
    "graph_dim": graph_dim,
    "target_dim": target_dim,
    "target_space": target_space,
    "capsules": len(capsule_ids),
    "has_path_supervision": has_path_supervision,
}


In [ ]:
training_slices = ACTIVE_DATASET.get("training_slices") or []
if training_slices:
    training_bundles = [
        get_dataset_adapter(ACTIVE_DATASET["adapter_name"], **slice_config).load(
            limit_documents=ACTIVE_DATASET.get("limit_documents"),
            limit_eval_samples=ACTIVE_DATASET.get("limit_eval_samples_per_slice"),
        )
        for slice_config in training_slices
    ]
else:
    training_bundles = [
        get_dataset_adapter(
            ACTIVE_DATASET["adapter_name"],
            **ACTIVE_DATASET.get("adapter_kwargs", {}),
        ).load(
            limit_documents=ACTIVE_DATASET.get("limit_documents"),
            limit_eval_samples=ACTIVE_DATASET.get("limit_eval_samples"),
        )
    ]
training_samples = [
    sample for loaded_bundle in training_bundles for sample in loaded_bundle.eval_samples
]
training_sample_counts = {}
for sample in training_samples:
    slice_name = str(sample.metadata.get("hop") or sample.metadata.get("type") or "default")
    training_sample_counts[slice_name] = training_sample_counts.get(slice_name, 0) + 1
random.Random(SEED).shuffle(training_samples)
bundle_metadata = [loaded_bundle.metadata for loaded_bundle in training_bundles]
graph_report = rgcn_metadata.get("graph_report") or {}
capsule_search_text_by_id = dict(zip(capsule_ids, capsule_search_texts))
if len(capsule_search_text_by_id) != len(capsule_ids):
    raise RuntimeError("Duplicate capsule ids found in the R-GCN projection manifest.")
len(capsule_search_text_by_id), len(training_samples), training_sample_counts

In [ ]:
HASH_TEXT_DIM = 512


def stable_bucket(token: str, dim: int) -> int:
    digest = hashlib.md5(token.encode("utf-8")).hexdigest()
    return int(digest, 16) % dim


def hashing_text_vector(text: str, dim: int = HASH_TEXT_DIM) -> np.ndarray:
    vector = np.zeros(dim, dtype="float32")
    for token in TOKEN_RE.findall(text.lower()):
        vector[stable_bucket(token, dim)] += 1.0
    norm = np.linalg.norm(vector)
    return vector / max(norm, 1e-12)


def embed_query_texts(texts: list[str]) -> np.ndarray:
    if not USE_OLLAMA_QUERY_EMBEDDINGS:
        return np.vstack([hashing_text_vector(text) for text in texts]).astype("float32")

    vectors = []
    for start in tqdm(range(0, len(texts), EMBED_BATCH_SIZE), desc="embedding query texts"):
        batch = texts[start : start + EMBED_BATCH_SIZE]
        vectors.extend(embedding_adapter.embed_texts(batch).vectors)
    return np.asarray(vectors, dtype="float32")


miner = (
    lambda: mine_metaqa_path_projection_examples(
        training_samples, capsule_search_text_by_id, capsule_edge_ids_by_id, projection_graph_edges,
        capsule_node_ids_by_id=capsule_node_ids_by_id,
        max_examples=int(ACTIVE_DATASET.get("max_projection_examples", 20_000)),
        max_positives=int(ACTIVE_DATASET.get("max_positive_capsules", 16)),
        max_negatives=int(ACTIVE_DATASET.get("max_hard_negatives", 20)),
    )
    if ACTIVE_DATASET.get("adapter_name") == "metaqa" and has_path_supervision
    else lambda: mine_projection_examples(
        training_samples, capsule_search_text_by_id,
        max_examples=int(ACTIVE_DATASET.get("max_projection_examples", 20_000)),
        max_positives=int(ACTIVE_DATASET.get("max_positive_capsules", 16)),
        max_negatives=int(ACTIVE_DATASET.get("max_hard_negatives", 20)),
    )
)
projection_examples, mining_report = miner()
if not projection_examples:
    raise RuntimeError("No multi-positive projection examples were mined. Check evidence labels and capsule search text.")
len(projection_examples), mining_report, projection_examples[:1]

In [ ]:
query_texts = [example.question for example in projection_examples]
query_vectors = embed_query_texts(query_texts)
TEXT_DIM = query_vectors.shape[1]
max_positive_count = max(len(example.positive_capsule_ids) for example in projection_examples)
max_negative_count = max(len(example.hard_negative_capsule_ids) for example in projection_examples)
pos_indices = np.zeros((len(projection_examples), max_positive_count), dtype="int64")
pos_mask = np.zeros_like(pos_indices, dtype=bool)
neg_indices = np.zeros((len(projection_examples), max_negative_count), dtype="int64")
neg_mask = np.zeros_like(neg_indices, dtype=bool)
for row, example in enumerate(projection_examples):
    positives = [capsule_to_idx[capsule_id] for capsule_id in example.positive_capsule_ids]
    negatives = [capsule_to_idx[capsule_id] for capsule_id in example.hard_negative_capsule_ids]
    pos_indices[row, :len(positives)] = positives
    pos_mask[row, :len(positives)] = True
    neg_indices[row, :len(negatives)] = negatives
    neg_mask[row, :len(negatives)] = True

indices = np.arange(len(projection_examples))
np.random.shuffle(indices)
val_size = max(1, int(len(indices) * 0.1))
if val_size >= len(indices):
    val_size = 1
val_idx = indices[:val_size]
train_idx = indices[val_size:]

capsule_tensor = torch.tensor(capsule_target_embeddings, dtype=torch.float32, device=device)


def loader(idxs, batch_size=256, shuffle=True):
    ds = TensorDataset(
        torch.tensor(query_vectors[idxs], dtype=torch.float32),
        torch.tensor(pos_indices[idxs], dtype=torch.long),
        torch.tensor(pos_mask[idxs], dtype=torch.bool),
        torch.tensor(neg_indices[idxs], dtype=torch.long),
        torch.tensor(neg_mask[idxs], dtype=torch.bool),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


len(train_idx), len(val_idx)

In [ ]:
class ProjectionMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 256, dropout: float = 0.10):
        super().__init__()
        self.base = nn.Identity() if input_dim == output_dim else nn.Linear(input_dim, output_dim, bias=False)
        self.residual = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.residual[-1].weight)
        nn.init.zeros_(self.residual[-1].bias)

    def forward(self, x):
        return F.normalize(self.base(x) + self.residual(x), dim=-1)


model = ProjectionMLP(TEXT_DIM, target_dim).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
margin = 0.05
temperature = 0.10


def multi_positive_loss(q_vec, pos_idx, positive_mask, neg_idx, negative_mask):
    projected = model(q_vec.to(device))
    positive_mask = positive_mask.to(device)
    negative_mask = negative_mask.to(device)
    positive = capsule_tensor[pos_idx.to(device)]
    negative = capsule_tensor[neg_idx.to(device)]
    pos_sim = torch.einsum("bd,bpd->bp", projected, positive)
    neg_sim = torch.einsum("bd,bnd->bn", projected, negative)
    pos_logits = (pos_sim / temperature).masked_fill(~positive_mask, float("-inf"))
    neg_logits = ((neg_sim + margin) / temperature).masked_fill(~negative_mask, float("-inf"))
    positive_mass = torch.logsumexp(pos_logits, dim=1)
    total_mass = torch.logsumexp(torch.cat([pos_logits, neg_logits], dim=1), dim=1)
    loss = (total_mass - positive_mass).mean()
    mean_positive = pos_sim[positive_mask].mean()
    hardest_negative = neg_sim.masked_fill(~negative_mask, float("-inf")).max(dim=1).values.mean()
    return loss, mean_positive, hardest_negative


@torch.no_grad()
def evaluate():
    model.eval()
    losses, pos_sims, neg_sims = [], [], []
    reciprocal_ranks, hits1, hits5, hits10, hits20 = [], [], [], [], []
    for q, pos, positive_mask, neg, negative_mask in loader(val_idx, shuffle=False):
        loss, pos_sim, neg_sim = multi_positive_loss(q, pos, positive_mask, neg, negative_mask)
        losses.append(float(loss.cpu()))
        pos_sims.append(float(pos_sim.cpu()))
        neg_sims.append(float(neg_sim.cpu()))
        projected = model(q.to(device))
        positive = capsule_tensor[pos.to(device)]
        positive_scores = torch.einsum("bd,bpd->bp", projected, positive)
        positive_scores = positive_scores.masked_fill(~positive_mask.to(device), float("-inf"))
        best_positive_scores = positive_scores.max(dim=1).values
        ranks = torch.ones(projected.shape[0], dtype=torch.long, device=device)
        for start in range(0, capsule_tensor.shape[0], CAPSULE_SCORE_BATCH_SIZE):
            capsule_batch = capsule_tensor[start : start + CAPSULE_SCORE_BATCH_SIZE]
            scores = projected @ capsule_batch.T
            ranks += (scores > best_positive_scores.unsqueeze(1)).sum(dim=1)
        reciprocal_ranks.extend((1.0 / ranks.float()).cpu().tolist())
        hits1.extend((ranks <= 1).float().cpu().tolist())
        hits5.extend((ranks <= 5).float().cpu().tolist())
        hits10.extend((ranks <= 10).float().cpu().tolist())
        hits20.extend((ranks <= 20).float().cpu().tolist())
    return {
        "val_loss": float(np.mean(losses)),
        "val_pos": float(np.mean(pos_sims)),
        "val_neg": float(np.mean(neg_sims)),
        "val_any_positive_mrr": float(np.mean(reciprocal_ranks)),
        "val_any_positive_recall_at_1": float(np.mean(hits1)),
        "val_any_positive_recall_at_5": float(np.mean(hits5)),
        "val_any_positive_recall_at_10": float(np.mean(hits10)),
        "val_any_positive_recall_at_20": float(np.mean(hits20)),
    }

In [ ]:
initial_validation = evaluate()
best_val = initial_validation["val_loss"]
best_mrr = initial_validation["val_any_positive_mrr"]
best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
patience = 7
bad_epochs = 0
history = [{"epoch": 0, "train_loss": None, **initial_validation}]
print(f"epoch=000 identity_baseline val={best_val:.4f} any_mrr={best_mrr:.3f} any_recall10={initial_validation['val_any_positive_recall_at_10']:.3f}")

for epoch in range(1, 101):
    model.train()
    train_losses = []
    for q, pos, positive_mask, neg, negative_mask in loader(train_idx):
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = multi_positive_loss(q, pos, positive_mask, neg, negative_mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(float(loss.detach().cpu()))
    validation = evaluate()
    val_loss = validation["val_loss"]
    train_loss = float(np.mean(train_losses)) if train_losses else 0.0
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **validation,
        }
    )
    print(f"epoch={epoch:03d} train={train_loss:.4f} val={val_loss:.4f} any_mrr={validation['val_any_positive_mrr']:.3f} any_recall10={validation['val_any_positive_recall_at_10']:.3f}")
    if validation["val_any_positive_mrr"] > best_mrr + 1e-4:
        best_val = val_loss
        best_mrr = validation["val_any_positive_mrr"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("early stopping")
            break

if best_state is not None:
    model.load_state_dict(best_state)
{"best_val_loss": best_val, "best_val_any_positive_mrr": best_mrr}

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), EXPORT_DIR / "query_projection_mlp.pt")
with (EXPORT_DIR / "query_projection_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "dataset": ACTIVE_DATASET,
            "bundle_metadata": bundle_metadata,
            "graph_report": graph_report,
            "input_dim": TEXT_DIM,
            "output_dim": target_dim,
            "structural_output_dim": graph_dim,
            "projection_examples": len(projection_examples),
            "projection_example_counts": mining_report["slice_counts"],
            "label_mining": mining_report,
            "training_sample_counts": training_sample_counts,
            "history": history,
            "best_val_loss": float(best_val),
            "best_val_any_positive_mrr": float(best_mrr),
            "capsule_fingerprint": expected_capsule_fingerprint,
            "training_schema_version": 5,
            "objective": "path_supervised_multi_positive_semantic_contrastive",
            "architecture": "identity_residual_mlp",
            "target_space": target_space,
            "target_manifest": target_manifest,
            "objective_margin": margin,
            "objective_temperature": temperature,
            "validation_rule": "best_rank_of_any_retained_positive_capsule",
            "supervision_rule": mining_report["label_rule"],
            "structural_artifact_name": ACTIVE_DATASET.get("structural_artifact_name", ACTIVE_DATASET["artifact_name"]),
            "projection_manifest": projection_manifest,
            "text_encoder": EMBEDDING_MODEL if USE_OLLAMA_QUERY_EMBEDDINGS else "stable_hashing_vectorizer_notebook",
        },
        f,
        indent=2,
    )
EXPORT_DIR